In [1]:
import torch
import numpy as np
import pandas as pd

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from datasets import DatasetDict, Dataset

import warnings
warnings.filterwarnings('ignore', message = 'Importing display from IPython.core.display is deprecated since IPython 7.14.*')
warnings.filterwarnings('ignore', message = """
    There is an imbalance between your GPUs.*""")
warnings.filterwarnings('ignore', message = 'Was asked to gather along dimension 0, but all input tensors were scalars.*')

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,  DataCollatorWithPadding, TextClassificationPipeline

from scipy.special import softmax
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score
from netcal.metrics import ECE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import random
from transformers import set_seed
seed = 123

set_seed(seed)
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

import sys

model_to_refit = 'microsoft/deberta-v3-large'
pretrained_model_name_or_path = model_to_refit

data = pd.read_csv('../../data/processed/train.csv')
data['target'] = data['target'].astype('str').replace({'0': 'Inaccurate', '1': 'Accurate'})
data['text'] = data['combined_spellcheck'].astype('str')
data = data[['text', 'target']]

train = data.sample(frac = 0.80, random_state = seed)
val = data[~data.index.isin(train.index)]

test = pd.read_csv('../../data/processed/test.csv')
test['target'] = test['target'].astype('str').replace({'0': 'Inaccurate', '1': 'Accurate'})
test['text'] = test['combined_spellcheck'].astype('str')
test = test[['text', 'target']]

label2id = {
    'Inaccurate': 0,
    'Accurate': 1
    }

id2label = {
    0: 'Inaccurate',
    1: 'Accurate'
    }

ds_train = Dataset.from_dict({'text': train.text, 
                              'labels': train.target})

ds_val = Dataset.from_dict({'text': val.text, 
                            'labels': val.target})

ds_test = Dataset.from_dict({'text': test.text, 
                             'labels': test.target})

dataset_dict = DatasetDict({
    'train': ds_train, 
    'val': ds_val, 
    'test': ds_test
})

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path)

def encode (examples):
    tokenized_examples = tokenizer(examples['text'], return_token_type_ids = True)
    tokenized_examples['labels'] = [label2id[label] for label in examples['labels']]
    return tokenized_examples

dataset_dict_tokenized = dataset_dict.map(
    encode,
    batched = True,
    num_proc = os.cpu_count(),
    remove_columns = ['text']
    )

data_collator = DataCollatorWithPadding(tokenizer, padding = True)

if "ModernBERT" in pretrained_model_name_or_path:
    def model_init ():
        return AutoModelForSequenceClassification.from_pretrained(
            pretrained_model_name_or_path,
            num_labels = len(test.groupby('target').size()),
            id2label = id2label,
            label2id = label2id,
            torch_dtype = torch.bfloat16
            )
else:
    def model_init ():
        return AutoModelForSequenceClassification.from_pretrained(
            pretrained_model_name_or_path,
            num_labels = len(test.groupby('target').size()),
            id2label = id2label,
            label2id = label2id
            )
    
def compute_metrics (eval_pred):
    predictions, labels = eval_pred
    preds = [int(np.argmax(prediction)) for prediction in predictions]
    probs = softmax(predictions, axis = -1)[:, 1]
    return {'eval_auc': roc_auc_score(labels, probs),
            'eval_bal_acc': balanced_accuracy_score(labels, preds)}

def eval_bal_acc (metrics):
    return metrics['eval_bal_acc']

def eval_auc (metrics):
    return metrics['eval_auc']

/home/daved/miniforge3/envs/bert/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map (num_proc=8):   0%|          | 0/3138 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/785 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1731 [00:00<?, ? examples/s]

In [2]:
# refit model via True; otherwise, False
if True:
    best_trial = lambda: None
    best_trial.hyperparameters = {
        'learning_rate': 4e-05,
        'per_device_train_batch_size': 8,
        'num_train_epochs': 4
    }

best_trial.hyperparameters

{'learning_rate': 4e-05,
 'per_device_train_batch_size': 8,
 'num_train_epochs': 4}

In [3]:
final_args = TrainingArguments(
    'training_output',
    learning_rate = best_trial.hyperparameters['learning_rate'],
    per_device_train_batch_size = best_trial.hyperparameters['per_device_train_batch_size'],
    num_train_epochs = best_trial.hyperparameters['num_train_epochs'],
    per_device_eval_batch_size = 32,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True,
    #metric_for_best_model = 'eval_bal_acc',
    #greater_is_better = True,
    report_to = 'none'
)

final_trainer = Trainer(
    args = final_args,
    data_collator = data_collator,
    model_init = model_init,
    train_dataset = dataset_dict_tokenized['train'],
    eval_dataset = dataset_dict_tokenized['val'],
    compute_metrics = compute_metrics
)

model_checkpoint = 'model_refit'
final_trainer.train()
final_trainer.save_model(model_checkpoint)
tokenizer.save_pretrained(model_checkpoint)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[2025-01-11 03:40:32,419] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/daved/miniforge3/envs/bert/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/daved/miniforge3/envs/bert/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Auc,Bal Acc
1,No log,0.651072,0.676020,0.635057
2,0.676600,0.618075,0.703573,0.663695
3,0.615100,0.664712,0.692857,0.662512
4,0.564300,0.639608,0.715675,0.678325


('model_refit/tokenizer_config.json',
 'model_refit/special_tokens_map.json',
 'model_refit/spm.model',
 'model_refit/added_tokens.json',
 'model_refit/tokenizer.json')

In [4]:
data_test_dict = []

for i, _ in test.iterrows():
    data_test_dict.append({'text': test['text'][i]})

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels = len(test.groupby('target').size())
    ).to(device)

pipe = TextClassificationPipeline(model = model, tokenizer = tokenizer, top_k = None, device = device)

# temporary workaround for XLNet batch size issue
if str(model.base_model).find('XLNetModel') != -1:
    batch_size = 1
else:
    batch_size = 128

raw_probs = pipe(data_test_dict, batch_size = batch_size)

probs = np.array([[i['score'] for i in item if i['label'] == id2label[1]][0] for item in raw_probs])
preds = np.where(probs >= 0.5, 1, 0)

y_true = [label2id[x] for x in test['target']]

n_bins = 7
ece = np.round(ECE(bins = n_bins).measure(np.array(probs), np.array(y_true)), 3)

print('AUC:', np.round(roc_auc_score(y_true, probs), 3))
print('Balanced Accuracy:', np.round(balanced_accuracy_score(y_true, preds), 3))
print('Macro F1:', np.round(f1_score(y_true, preds, average = 'macro'), 3))
print('ECE', ece)

Device set to use cuda


AUC: 0.724
Balanced Accuracy: 0.682
Macro F1: 0.684
ECE 0.024
